# 10-5 字典在 APCS 的經典解題模式 (Dictionary Patterns in APCS)

在掌握了字典的建立、存取與常用方法後，我們終於要將字典應用在 APCS 程式設計實戰中了！許多初學者在解題時，習慣只用一維或二維串列解決所有問題，結果往往在遇到「超大數值範圍」或「非連續字串代碼」時，陷入記憶體超限（MLE）或超時（TLE）的泥淖。

本節將透過**極致緩坡架構**，深入剖析 APCS 考場最常出現的五大字典經典解題模式：動態頻率統計、符號查表法、座標壓縮離散化、雙向映射表、分組聚合（Bucket），並實測字典與串列在查詢效能上的巨大天壤之別。

### 🎯 本節學習目標
1. 掌握「頻率統計模式」，以動態稀疏計數突破陣列長度限制。
2. 熟練「查表法（Lookup Table）」，以字典取代冗長的巢狀分支條件。
3. 理解「離散化（座標壓縮）」的核心概念，以排名映射縮小超大座標空間。
4. 實現「雙向映射表」，兼顧正向與反向查詢的 $O(1)$ 極速效能。
5. 掌握「分組聚合（Group By / Bucket）」，以字典結合串列完成分類收集。
6. 實測 $O(1)$ 雜湊查詢與 $O(N)$ 線性走訪在巨量數據下的效能差異。

> ⚠️ **溫馨提醒**：本節程式碼完全不依賴自訂函式 `def` 或集合 `set`，請專注於字典本體在演算法中的思維轉換！

## 10.5.1 頻率統計模式（Frequency Counter）：大數值與字串動態計數器

### 觀念說明
在統計數據出現次數時，如果數值範圍很小（例如分數 0 到 100），我們可以使用一維串列 `counts = [0] * 101` 來記錄。
但是，如果題目給定的資料是：
1. **英文單字或代碼**（例如：統計文章中每個英文字出現的次數）。
2. **極大整數**（例如：數值高達 $10^9$ 或為負數）。

此時若嘗試建立長度十億的串列，記憶體會立刻爆炸引發 **MLE（Memory Limit Exceeded）**！
而字典就是解決這個問題的神器！字典採用**動態稀疏儲存**，只有實際出現過的數值才會在字典中佔用記憶體空間。
搭配我們在 10-4 學到的 `counts[x] = counts.get(x, 0) + 1`，無論輸入是字串還是十億等級的大數，都能以短短一行程式碼優雅完成統計，是 APCS 統計題型的標準黃金公式！

In [ ]:
# 10.5.1 範例：單字與超大數值動態頻率統計
tokens = ["apple", "banana", "apple", "orange", "banana", "apple"]

word_counts = {}
for word in tokens:
    word_counts[word] = word_counts.get(word, 0) + 1

print("單字出現頻率:", word_counts)

# 統計大數值（若用串列會 MLE，用字典毫無壓力）
big_numbers = [1000000000, -999999, 1000000000, 5, -999999, 1000000000]
num_counts = {}
for num in big_numbers:
    num_counts[num] = num_counts.get(num, 0) + 1

print()
print("大數值頻率統計:", num_counts)

In [ ]:
# 10.5.1 填空題：以 get() 補齊投票開票計數
votes = ["Alice", "Bob", "Alice", "Charlie", "Bob", "Alice"]

tally = {}
for candidate in votes:
    # 任務：若候選人尚未存在，初值視為 0，並將票數累加 1
    tally[candidate] = tally.get(candidate, 0) + 1

# 請將上方邏輯填入空格：
# tally[candidate] = tally.get(___, ___) + 1
print("最終計票結果:", tally)
# 預期輸出: 最終計票結果: {'Alice': 3, 'Bob': 2, 'Charlie': 1}

In [ ]:
# 10.5.1 練習題：最高票選拔統計機
# 題目說明：
# 給定投票清單 ballot_box，請統計每位候選人的得票數。
# 接著走訪計票字典，找出獲得最高票數的候選人與其得票數。
# （若最高票有多人，記錄第一位遇到的即可）
#
# 【公開測試資料 1】
# 投票清單: ["A", "B", "A", "C", "B", "A"]
# 輸出: 最高票候選人: A 得票數: 3
#
# 【公開測試資料 2】
# 投票清單: ["cat", "dog", "cat", "dog", "dog"]
# 輸出: 最高票候選人: dog 得票數: 3

ballot_box = ["A", "B", "A", "C", "B", "A"]

votes_map = {}
for name in ballot_box:
    votes_map[name] = votes_map.get(name, 0) + 1

winner = ""
max_votes = -1
for name, count in votes_map.items():
    if count > max_votes:
        max_votes = count
        winner = name

print("最高票候選人:", winner, "得票數:", max_votes)

In [ ]:
# 10.5.1 挑戰題：尋找唯一落單的幸運號碼
# 題目說明：
# 給定一個包含許多重複數字的串列 nums = [7, 3, 5, 3, 7, 9, 5]
# 其中除了一個神秘號碼只出現過剛好「1 次」外，其他號碼都出現了 2 次以上。
# 請使用字典頻率統計，找出並印出該唯一落單的號碼。

nums = [7, 3, 5, 3, 7, 9, 5]

# 請在下方寫出你的解答代碼
freq = {}
for n in nums:
    freq[n] = freq.get(n, 0) + 1

unique_num = None
for k, v in freq.items():
    if v == 1:
        unique_num = k
        break

print("唯一落單號碼:", unique_num)
# 預期結果: 唯一落單號碼: 9

## 10.5.2 查表法（Lookup Table）：英文代碼與符號轉換

### 觀念說明
在 APCS 許多題目中，需要將某種符號或英文代碼轉換為對應的數值或名稱。例如：
- 將英文月份 `"Jan"`, `"Feb"` 轉換為月份數字 `1`, `2`。
- 將羅馬數字 `"I"`, `"V"`, `"X"`, `"L"` 轉換為十進位數字 `1`, `5`, `10`, `50`。
- 將 APCS 對聯題的平仄符號轉換為數值代碼。

傳統的做法是寫一大串 `if-elif-elif-else` 分支，但這樣程式碼冗長難讀，且容易漏掉條件。
**查表法（Lookup Table）** 的核心思想就是：**用字典預先建立好「對應關係」**！查詢時直接呼叫 `table[key]` 或安全取值 `table.get(key, default)`，程式碼立刻縮短為一兩行，既直觀又完全不易出錯。

In [ ]:
# 10.5.2 範例：羅馬數字字串求和查表法
roman_table = {
    "I": 1,
    "V": 5,
    "X": 10,
    "L": 50,
    "C": 100
}

roman_str = "XVI"  # 10 + 5 + 1 = 16

total_value = 0
for ch in roman_str:
    total_value += roman_table.get(ch, 0)

print("羅馬數字:", roman_str, "換算結果:", total_value)

In [ ]:
# 10.5.2 填空題：月份縮寫轉數字查表
month_map = {
    "Jan": 1, "Feb": 2, "Mar": 3, "Apr": 4,
    "May": 5, "Jun": 6, "Jul": 7, "Aug": 8
}

target_month = "May"
# 任務：使用查表法取得 target_month 的月份數字，查不到回傳 -1
month_num = month_map.get(target_month, -1)

# 請填入空格：
# month_num = month_map.get(___, ___)
print(target_month, "對應月份數字:", month_num)
# 預期輸出: May 對應月份數字: 5

In [ ]:
# 10.5.2 練習題：外幣匯率試算器
# 題目說明：
# 給定新台幣兌換外幣之匯率字典 exchange_rates（表示 1 單位外幣可換多少新台幣）。
# 給定欲兌換的交易紀錄 transactions，每筆紀錄為 (幣別, 金額) 之元組。
# 請計算出所有交易紀錄折合新台幣的總金額（取整數）。
#
# 【公開測試資料 1】
# 匯率表: {"USD": 31, "JPY": 0.21, "EUR": 34}
# 交易清單: [("USD", 100), ("JPY", 10000), ("EUR", 50)]
# 計算: 100*31 + 10000*0.21 + 50*34 = 3100 + 2100 + 1700 = 6900
# 輸出: 折合新台幣總額: 6900 元
#
# 【公開測試資料 2】
# 匯率表: {"USD": 30, "KRW": 0.025}
# 交易清單: [("USD", 10), ("KRW", 20000)]
# 計算: 300 + 500 = 800
# 輸出: 折合新台幣總額: 800 元

exchange_rates = {"USD": 31, "JPY": 0.21, "EUR": 34}
transactions = [("USD", 100), ("JPY", 10000), ("EUR", 50)]

total_twd = 0.0
for currency, amount in transactions:
    rate = exchange_rates.get(currency, 0.0)
    total_twd += amount * rate

print("折合新台幣總額:", int(total_twd), "元")

In [ ]:
# 10.5.2 挑戰題：英文星期天數差計算
# 題目說明：
# 給定星期英文對照字典 weekday_map = {"Mon": 1, "Tue": 2, "Wed": 3, "Thu": 4, "Fri": 5, "Sat": 6, "Sun": 7}
# 輸入兩個星期的縮寫 day1 與 day2（例如 "Mon" 與 "Fri"）。
# 請計算從 day1 到 day2 經過的天數（若 day2 在 day1 之前，表示跨到下一週，需補 7 天）。
# 例如: day1 = "Fri" (5), day2 = "Mon" (1) -> 跨週差 (1 - 5 + 7) % 7 = 3 天。

weekday_map = {"Mon": 1, "Tue": 2, "Wed": 3, "Thu": 4, "Fri": 5, "Sat": 6, "Sun": 7}
day1 = "Fri"
day2 = "Mon"

# 請在下方寫出你的解答代碼
d1 = weekday_map[day1]
d2 = weekday_map[day2]
diff = (d2 - d1) % 7
if diff == 0 and day1 != day2:
    diff = 7

print("經過天數:", diff)
# 預期結果: 經過天數: 3

## 10.5.3 離散化 / 座標壓縮（Coordinate Compression）：以字典建立排名映射

### 觀念說明
在 APCS 進階題目中，常常會遇到「數值極大但數量很少」的情況。例如：
數列中有 5 個數字：`[1000000000, 50, 99999999, 50, 123456]`。
雖然這些數字高達十億，但它們彼此之間其實只在乎**「相對大小與排名」**！

此時，「**離散化（座標壓縮）**」就是核心技巧：
1. 將原始數列提取出來，進行**排序並去除重複值**。
2. 使用字典建立一個**排名映射表**：`rank_map[數值] = 排名（從 0 或 1 開始）`。
3. 將原始數列中的每個數值替換成它的排名！

這樣一來，高達十億的巨大數值就被無痛壓縮成了 `0, 1, 2, 3`，後續就能直接放進普通陣列中進行標記，徹底化解 MLE 的危機！

In [ ]:
# 10.5.3 範例：學生成績相對排名離散化
scores = [88, 95, 70, 88, 100, 70]
print("原始分數:", scores)

# 1. 排序並去重（不使用 set，純用 sorted 與迴圈去重）
sorted_scores = sorted(scores)
unique_sorted = []
for s in sorted_scores:
    if len(unique_sorted) == 0 or s != unique_sorted[-1]:
        unique_sorted.append(s)

print("排序去重後:", unique_sorted)

# 2. 使用字典建立「數值 -> 排名（從 0 開始）」映射表
rank_map = {}
for idx in range(len(unique_sorted)):
    val = unique_sorted[idx]
    rank_map[val] = idx

print("排名對照字典:", rank_map)

# 3. 將原串列離散化成相對名次
compressed_scores = [rank_map[s] for s in scores]
print("離散化結果:", compressed_scores)

In [ ]:
# 10.5.3 填空題：建立座標壓縮字典
unique_coords = [105, 500, 1000, 99999]

coord_to_rank = {}
# 任務：走訪 unique_coords 的索引與數值，建立數值到排名的映射
for rank in range(len(unique_coords)):
    pos = unique_coords[rank]
    coord_to_rank[pos] = rank

# 請填入空格：
# coord_to_rank[___] = ___
print("座標壓縮表:", coord_to_rank)
# 預期輸出: 座標壓縮表: {105: 0, 500: 1, 1000: 2, 99999: 3}

In [ ]:
# 10.5.3 練習題：選手背號稠密排名轉換器
# 題目說明：
# 給定選手抵達終點的背號順序 runner_ids。
# 請將背號按小到大排序並去重，建立「背號 -> 緊湊索引 (0, 1, 2...)」的字典。
# 最後將 runner_ids 中的每個背號轉換成緊湊索引並印出新串列。
#
# 【公開測試資料 1】
# 背號序列: [805, 102, 550, 102, 805]
# 排序去重: [102, 550, 805] -> 映射 {102: 0, 550: 1, 805: 2}
# 輸出: 壓縮結果: [2, 0, 1, 0, 2]
#
# 【公開測試資料 2】
# 背號序列: [99, 10, 50]
# 輸出: 壓縮結果: [2, 0, 1]

runner_ids = [805, 102, 550, 102, 805]

# 1. 排序並去重
sorted_unique = []
for x in sorted(runner_ids):
    if len(sorted_unique) == 0 or x != sorted_unique[-1]:
        sorted_unique.append(x)

# 2. 建立字典
id_map = {}
for i in range(len(sorted_unique)):
    id_map[sorted_unique[i]] = i

# 3. 轉換輸出
compressed = [id_map[x] for x in runner_ids]
print("壓縮結果:", compressed)

In [ ]:
# 10.5.3 挑戰題：離散化反向解碼器
# 題目說明：
# 給定壓縮後的數列 compressed = [1, 0, 2, 0] 以及映射字典 rank_to_val = {0: 1000, 1: 5000, 2: 9999}
# 請透過字典查詢，將 compressed 數列還原回原本的原始數值串列並輸出。

compressed = [1, 0, 2, 0]
rank_to_val = {0: 1000, 1: 5000, 2: 9999}

# 請在下方寫出你的解答代碼
original_nums = [rank_to_val[r] for r in compressed]

print("還原原始數列:", original_nums)
# 預期結果: 還原原始數列: [5000, 1000, 9999, 1000]

## 10.5.4 雙向映射表（Two-way Mapping）：Key 與 Value 互換技巧

### 觀念說明
在某些通訊或加密題目中，我們常常需要同時支援「雙向查詢」：
- 給「學生學號」要能查出「學生姓名」。
- 給「學生姓名」也要能查出「學生學號」。

如果只建立一個 `id_to_name` 字典，當要用姓名反查學號時，就必須跑一個 for 迴圈逐一比對每個 `value`，時間複雜度高達 $O(N)$。如果有 $M$ 次查詢，總時間就是 $O(M \times N)$，在測資龐大時極易超時。

### 雙向字典解決方案
最佳的作法是**同時維護兩個字典**，或者在讀取資料後，利用一個迴圈走訪 `for k, v in d.items():` 自動生成反向字典 `inv_d[v] = k`（前提是值必須唯一）。這樣不論由前查後、還是由後查前，每一次查詢都是瞬間完成的 **$O(1)$**！

In [ ]:
# 10.5.4 範例：工號與姓名雙向極速查詢
id_to_name = {
    "E101": "Alice",
    "E102": "Bob",
    "E103": "Charlie"
}

# 自動產生反向字典 name_to_id
name_to_id = {name: emp_id for emp_id, name in id_to_name.items()}

print("正向字典 (工號查姓名):", id_to_name)
print("反向字典 (姓名查工號):", name_to_id)

# 雙向皆為 O(1) 瞬間查表
query_id = "E102"
print(query_id, "的姓名是:", id_to_name.get(query_id))

query_name = "Charlie"
print(query_name, "的工號是:", name_to_id.get(query_name))

In [ ]:
# 10.5.4 填空題：反向字典建置
morse_encode = {"A": ".-", "B": "-...", "C": "-.-."}

morse_decode = {}
# 任務：使用 items() 走訪 morse_encode，將摩斯密碼作為鍵，字母作為值存入 morse_decode
for letter, code in morse_encode.items():
    morse_decode[code] = letter

# 請填入空格：
# morse_decode[___] = ___
print("摩斯解碼字典:", morse_decode)
# 預期輸出: 摩斯解碼字典: {'.-': 'A', '-...': 'B', '-.-.': 'C'}

In [ ]:
# 10.5.4 練習題：情報員暗號雙向翻譯機
# 題目說明：
# 現有一份特務代號對照表 agent_codes，鍵為「真名」，值為「代號」。
# 請建置好雙向字典。
# 給定一連串查詢指令 queries，格式為 (查詢模式, 查詢目標)：
# - 若模式為 "CODE"：表示目標為真名，請輸出其代號。
# - 若模式為 "NAME"：表示目標為代號，請輸出其真名。
#
# 【公開測試資料 1】
# 對照表: {"Bond": "007", "Hunt": "IMF01", "Bourne": "T100"}
# 查詢: [("CODE", "Bond"), ("NAME", "IMF01")]
# 輸出:
# 007
# Hunt
#
# 【公開測試資料 2】
# 對照表: {"Neo": "ONE", "Trinity": "TWO"}
# 查詢: [("NAME", "ONE"), ("CODE", "Trinity")]
# 輸出:
# Neo
# TWO

agent_codes = {"Bond": "007", "Hunt": "IMF01", "Bourne": "T100"}
queries = [("CODE", "Bond"), ("NAME", "IMF01")]

# 建立反向字典
code_to_name = {c: n for n, c in agent_codes.items()}

for mode, target in queries:
    if mode == "CODE":
        print(agent_codes[target])
    elif mode == "NAME":
        print(code_to_name[target])

In [ ]:
# 10.5.4 挑戰題：一對多反向分組索引
# 題目說明：
# 給定學生成績字典 score_dict = {"Alice": 90, "Bob": 80, "Charlie": 90, "David": 70, "Eva": 80}
# 注意：有多個學生的成績可能相同！
# 請建立一個反向字典 score_to_names，鍵為「分數」，值為「擁有該分數的學生姓名串列（list）」。

score_dict = {"Alice": 90, "Bob": 80, "Charlie": 90, "David": 70, "Eva": 80}

# 請在下方寫出你的解答代碼
score_to_names = {}
for name, score in score_dict.items():
    if score not in score_to_names:
        score_to_names[score] = []
    score_to_names[score].append(name)

print("分數反向索引:", score_to_names)
# 預期結果: 分數反向索引: {90: ['Alice', 'Charlie'], 80: ['Bob', 'Eva'], 70: ['David']}

## 10.5.5 分組聚合（Group By / Bucket）：以字典收集分類清單

### 觀念說明
在上一題挑戰題中，我們遇到了一個重要的結構：**字典的值（Value）可以是串列（List）**！
這就是 APCS 與演算法中極其高頻的**「分組聚合（Bucket / Group By）」模式**。

### 運作模式
當我們想要將大量資料依照某個特徵（例如：縣市、年級、奇偶數、單字首字母）進行分類收集時：
1. 建立一個空字典 `groups = {}`。
2. 走訪每筆資料，取出其分類標籤 `cat`。
3. **安全初值初始化**：若 `cat not in groups:`，先建立一個空串列 `groups[cat] = []`。
4. **收集歸類**：使用 `groups[cat].append(item)` 將資料放入該分類的專屬籃子中！

這個技巧不僅在資料統計中不可或缺，在後續章節學習**圖論鄰接串列（Adjacency List）**時，更是表示節點連接關係的絕對主力架構！

In [ ]:
# 10.5.5 範例：依年級收集學生名單
students = [
    (7, "Alice"),
    (8, "Bob"),
    (7, "Charlie"),
    (9, "David"),
    (8, "Eva")
]

grade_buckets = {}
for grade, name in students:
    # 若該年級尚未有籃子，先建立空串列
    if grade not in grade_buckets:
        grade_buckets[grade] = []
    # 將學生姓名加入該年級的籃子
    grade_buckets[grade].append(name)

print("各年級學生分組結果:")
for grade in sorted(grade_buckets.keys()):
    print(f"{grade} 年級名冊:", grade_buckets[grade])

In [ ]:
# 10.5.5 填空題：奇偶數分組收集
numbers = [12, 7, 19, 24, 30, 15]

parity_groups = {"EVEN": [], "ODD": []}
for num in numbers:
    if num % 2 == 0:
        parity_groups["EVEN"].append(num)
    else:
        parity_groups["ODD"].append(num)

# 請將上方 append 填入空格：
# parity_groups["EVEN"].append(___)
# parity_groups["ODD"].append(___)
print("偶數群組:", parity_groups["EVEN"])
print("奇數群組:", parity_groups["ODD"])
# 預期輸出:
# 偶數群組: [12, 24, 30]
# 奇數群組: [7, 19, 15]

In [ ]:
# 10.5.5 練習題：單字首字母分組歸檔器
# 題目說明：
# 給定單字串列 words。
# 請將所有單字依照其「第一個英文字母（大寫）」進行分組。
# 分組完成後，依字母順序（A 到 Z）輸出每個字母及其收集到的單字串列。
#
# 【公開測試資料 1】
# 單字串列: ["apple", "ant", "banana", "bear", "cat"]
# 輸出:
# A : ['apple', 'ant']
# B : ['banana', 'bear']
# C : ['cat']
#
# 【公開測試資料 2】
# 單字串列: ["dog", "duck", "elephant"]
# 輸出:
# D : ['dog', 'duck']
# E : ['elephant']

words = ["apple", "ant", "banana", "bear", "cat"]

letter_buckets = {}
for w in words:
    first_char = w[0].upper()
    if first_char not in letter_buckets:
        letter_buckets[first_char] = []
    letter_buckets[first_char].append(w)

for ch in sorted(letter_buckets.keys()):
    print(ch, ":", letter_buckets[ch])

In [ ]:
# 10.5.5 挑戰題：餘數分桶總和統計
# 題目說明：
# 給定正整數串列 data = [10, 14, 21, 25, 33, 42, 50] 以及除數 k = 3
# 請將所有數字依據「除以 k 的餘數 (num % k)」進行分組（0, 1, 2 共三組）。
# 分組後，請印出每個餘數分組中所有數字的「總和（sum）」。

data = [10, 14, 21, 25, 33, 42, 50]
k = 3

# 請在下方寫出你的解答代碼
rem_buckets = {0: [], 1: [], 2: []}
for x in data:
    rem = x % k
    rem_buckets[rem].append(x)

for rem in range(k):
    total = sum(rem_buckets[rem])
    print(f"餘數 {rem} 總和:", total)
# 預期結果:
# 餘數 0 總和: 96
# 餘數 1 總和: 85
# 餘數 2 總和: 14

## 10.5.6 字典與串列效能實測對比：O(1) vs O(N) 查詢速度震撼體驗

### 觀念說明
在 APCS 考場上，我們常常看到同樣邏輯的程式，有人拿到滿分 AC，有人卻拿到超時 TLE。關鍵往往就在於**容器的選擇**！

### 查詢複雜度大對決
- **串列（List）**：使用 `x in my_list` 時，電腦必須從第 0 格開始依序往後比對。若串列有一萬個元素，最差情況要比對一萬次，時間複雜度為 **$O(N)$**。
- **字典（Dict）**：使用 `x in my_dict` 時，電腦利用雜湊函數直接計算出記憶體地址，一步到位，平均時間複雜度為 **$O(1)$**！

如果我們要執行 $M$ 次查詢，串列需要花費 $O(M \times N)$ 的時間；而字典只需要 $O(M)$！
當 $N = 100,000$ 且 $M = 100,000$ 時，串列需要執行一百億次比對（保證超時崩潰）；而字典只需要十萬次計算（約 0.05 秒搞定）！這就是雜湊技術帶來的降維打擊。

In [ ]:
# 10.5.6 範例：比對步數模擬與震撼實測
# 模擬 10,000 筆資料的查詢比對步數
data_size = 10000
target = 9999  # 放在最末端

# 1. 串列走訪查詢：必須逐一比對
list_steps = target + 1  # 從 0 到 9999 需比對 10000 次
print(f"串列走訪至目標 {target}，共需比對步數:", list_steps)

# 2. 字典雜湊查詢：直接計算 Hash 索引
dict_steps = 1
print(f"字典雜湊至目標 {target}，共需比對步數:", dict_steps)

print(f"效能差異倍數: 字典比串列快上了 {list_steps // dict_steps} 倍！")

In [ ]:
# 10.5.6 填空題：以字典建立極速過濾白名單
whitelist_users = ["alice", "bob", "charlie", "david"]

# 任務：將串列轉換為字典，使得查詢時間從 O(N) 降為 O(1)
whitelist_dict = {user: True for user in whitelist_users}

query_user = "charlie"
is_valid = query_user in whitelist_dict

# 請填入空格：
# is_valid = query_user ___ whitelist_dict
print(query_user, "是否在白名單中:", is_valid)
# 預期輸出: charlie 是否在白名單中: True

In [ ]:
# 10.5.6 練習題：極速資料去重過濾（O(N) 雜湊法）
# 題目說明：
# 給定一個長度很長且可能包含重複元素的串列 raw_stream。
# 請在「保留原本元素第一次出現的順序」前提下，將重複的元素剔除。
# 要求：不可使用 list.count()（因為會退化成 O(N^2)），
# 必須使用字典 seen_dict 來以 O(1) 判定是否已經出現過！
#
# 【公開測試資料 1】
# 串流資料: [4, 2, 4, 3, 2, 5, 1, 3]
# 去重結果: [4, 2, 3, 5, 1]
#
# 【公開測試資料 2】
# 串流資料: ["apple", "banana", "apple", "orange"]
# 去重結果: ['apple', 'banana', 'orange']

raw_stream = [4, 2, 4, 3, 2, 5, 1, 3]

seen_dict = {}
unique_list = []

for item in raw_stream:
    if item not in seen_dict:
        seen_dict[item] = True
        unique_list.append(item)

print("去重結果:", unique_list)

In [ ]:
# 10.5.6 挑戰題：兩數串列交集元素極速提取（O(N + M) 解法）
# 題目說明：
# 給定兩個串列 list_a = [1, 3, 5, 7, 9] 與 list_b = [2, 3, 6, 7, 10]
# 請不要使用雙重迴圈（O(A * B)），而是先將 list_a 的所有元素放入字典 table_a 中，
# 接著走訪 list_b，若元素出現在 table_a 中，則將其加入共同交集清單 intersection 中。
# （保證兩個串列內部元素皆不重複）

list_a = [1, 3, 5, 7, 9]
list_b = [2, 3, 6, 7, 10]

# 請在下方寫出你的解答代碼
table_a = {x: True for x in list_a}
intersection = [y for y in list_b if y in table_a]

print("共同交集元素:", intersection)
# 預期結果: 共同交集元素: [3, 7]

## 10.5 學習總結與核心技能檢核

太棒了！你已經全面掌握了 APCS 中字典的五大經典解題模式。

### 經典模式速查表
| 解題模式 | 核心原理與語法 | APCS 實戰優勢與破解情境 |
| :--- | :--- | :--- |
| **動態頻率統計** | `counts[x] = counts.get(x, 0) + 1` | 破解十億級數值與字串計數，徹底杜絕 MLE 記憶體超限 |
| **查表法 (Lookup)**| `val = table.get(key, default)` | 取代又長又臭的 `if-elif`，程式碼精準直觀不易出錯 |
| **離散化 (座標壓縮)**| `rank_map[v] = rank` | 將稀疏巨大座標壓縮為連續稠密整數，方便陣列索引 |
| **雙向映射表** | 同時維護 `id_to_name` 與 `name_to_id` | 雙向查詢皆保證 $O(1)$ 速度，避免反查退化為 $O(N)$ |
| **分組聚合 (Bucket)**| `groups[cat].append(item)` | 字典套串列，各類別專屬分類收集，為圖論鄰接串列打底 |
| **$O(1)$ 雜湊查詢** | `item in my_dict` | 雜湊運算一步到位，十萬筆資料瞬間過濾防禦 TLE |

至此，字典的所有強大威力你已瞭然於胸！從下一節 10-6 開始，我們將解鎖 Python 的第三個重要雜湊容器——**集合（Set）**，學習天然去重與集合運算的極致美學！